# Metodos de Simulacion - Práctica Primera Parte
## Assignment: Generación de Variables Aleatorias - IMSL Generator
- Alumno 1: Luis Carlés Durá
- Alumno 2: Juan Rodriguez Díaz de Greñu
- Alumno 3: Eder Tarifa Fernandez
- Alumno 4: Xiaoyuan Xu

### Base Code for Lehmer Generator by Luis Carles Dura

### 1. Generador congruencial multiplicativo de IMSL

Para la generación de números pseudoaleatorios utilizaremos un **generador congruencial multiplicativo** (*Multiplicative Congruential Generator*), siguiendo los parámetros establecidos por IMSL.

Este tipo de generador produce una secuencia determinista de números enteros mediante una relación de recurrencia. Cada nuevo valor se obtiene multiplicando el estado anterior por una constante y calculando posteriormente el resto de la división por un módulo.

La expresión matemática que define el generador es:

$$
X_{n+1}=(a\cdot X_n)\bmod m
$$

Donde:

- $X_n$ representa el estado actual del generador.
- $X_{n+1}$ es el siguiente estado de la secuencia.
- $a$ es la constante multiplicativa.
- $m$ es el módulo.
- $X_0$ es la semilla inicial (*seed*), que determina la secuencia generada.

En nuestro caso, utilizaremos los siguientes parámetros:

$$
\boxed{a=16807}
$$

$$
\boxed{m=2^{31}-1=2147483647}
$$

Una vez obtenido cada nuevo estado $X_n$, lo normalizaremos dividiéndolo por el módulo para obtener números pseudoaleatorios comprendidos en el intervalo $[0,1)$:

$$
U_n=\frac{X_n}{m}
$$

El objetivo es obtener una muestra de valores que se comporte estadísticamente como una sucesión de variables aleatorias independientes con distribución uniforme:

$$
U_n\sim \mathcal{U}(0,1)
$$

Es importante señalar que, aunque los valores generados son deterministas y dependen de la semilla inicial, esperamos que presenten propiedades estadísticas similares a las de una muestra verdaderamente aleatoria. Precisamente, estas propiedades serán evaluadas posteriormente mediante los contrastes de la batería Diehard.

In [9]:
def imsl_generator(N, seed):
    """
    Generador congruencial multiplicativo IMSL.

    """

    a = 16807
    m = 2**31 - 1

    if not isinstance(seed, int) or not (1 <= seed < m):
        raise ValueError("non valid seed.")

    X = seed
    random_numbers = []

    for _ in range(N):

        X = (a * X) % m

        U = X / m

        random_numbers.append(U)

    return random_numbers

### 2. Semilla Inicial y generación de una muestra
Se genera de forma determinística una muestra de numeros aleatorios con el generador imsl. Una muestra de tamaño N y de semilla (seed).

**PD:** para evitar que el generador produzca permanentemente una secuencia de ceros, la semilla debe ser un entero comprendido entre $1$ y $m-1$.

In [54]:
# Parameters
N = 12_000_000
seed = 12345

In [55]:
# Generate random sample of numbers using Lehmer generator
sample = imsl_generator(N, seed)

print("First 10 generated numbers:")
print(sample[:10])

print("Smaple Size:", len(sample))
print("Min:", min(sample))
print("Max:", max(sample))

First 10 generated numbers:
[0.09661652850760917, 0.8339946273872604, 0.9477024976851895, 0.035878594981449935, 0.011545853229028104, 0.051155220275351417, 0.7657871678312249, 0.5849297393974521, 0.9141300529773021, 0.7838003895170057]
Smaple Size: 12000000
Min: 2.0023435363556927e-08
Max: 0.9999998649582266


### 3. Preparación y exportación de la muestra para Dieharder

Una vez generada la muestra de números pseudoaleatorios mediante el generador congruencial multiplicativo de IMSL, debemos almacenarla en un formato que permita su posterior análisis mediante **Dieharder**, una herramienta que implementa diferentes contrastes estadísticos para evaluar la calidad de los generadores de números aleatorios.

Dieharder admite diferentes formatos de entrada. En nuestro caso, utilizaremos el formato **ASCII (tipo 202)**, que permite proporcionar directamente una secuencia de valores numéricos mediante un fichero de texto.
La cabecera del fichero incluirá los siguientes metadatos:

- **type:** identifica el tipo de datos proporcionados. Utilizaremos `d` para indicar que se trata de valores decimales.
- **count:** indica el número total de valores incluidos en la muestra.
- **numbit:** especifica el tamaño de palabra utilizado por el generador, expresado en bits.

A continuación, se almacenará cada número pseudoaleatorio en una línea independiente, utilizando una representación decimal con 15 cifras después del separador decimal.

La estructura general del fichero será la siguiente:

```text
# IMSL Multiplicative Congruential Generator
# Multiplier: 16807
# Modulus: 2147483647
# Seed: 12345
type: d
count: 1000000
numbit: 32
0.096616528507609
0.833994627387261
0.947702497660836
...
```

De esta manera, obtenemos un fichero de texto que puede utilizarse como entrada para los contrastes estadísticos de Dieharder, tambien se genera el archivo en formato binario.
Una vez creado el fichero, podremos utilizarlo como entrada de Dieharder mediante la opción `-g 201`, que identifica el formato binario.

In [56]:
def save_sample(sample, seed, filename="imsl_sample.txt"):

    with open(filename, "w", encoding="ascii") as file:

        file.write("# IMSL Multiplicative Congruential Generator\n")
        file.write(f"# Multiplier: 16807\n")
        file.write(f"# Modulus: {2**31 - 1}\n")
        file.write(f"# Seed: {seed}\n")

        file.write("type: g\n")
        file.write(f"count: {len(sample)}\n")
        file.write("numbit: 32\n")

        for number in sample:
            file.write(f"{number:.15f}\n")

In [57]:
# Prepare .txt file for Dieharder
save_sample(sample, seed)

In [58]:
import numpy as np

def save_binary_sample(random_numbers, filename="imsl_sample.bin"):
    values = np.asarray(random_numbers, dtype=np.float64)

    # Convertir U(0,1) a enteros de 32 bits
    integers = (values * (2**32)).astype(np.uint32)

    # Guardar en formato binario
    integers.tofile(filename)

In [59]:
# Prepare .bin file for Dieharder
save_binary_sample(sample)

### Instalar Dieharder
sudo apt update

sudo apt install dieharder

### 4. Análisis mediante contrastes estadísticos de Dieharder


In [61]:
import subprocess

result = subprocess.run(
    #["dieharder", "-g", "201", "-f", "imsl_sample.bin", "-d", "0"],
    ["dieharder", "-g", "201", "-f", "imsl_sample.bin", "-a"],
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result.stdout)

print("STDERR:")
print(result.stderr)

print("Return code:", result.returncode)

STDOUT:
#=============================================================================#
#            dieharder version 3.31.1 Copyright 2003 Robert G. Brown          #
#=============================================================================#
   rng_name    |           filename             |rands/second|
 file_input_raw|                 imsl_sample.bin|  4.04e+07  |
#=============================================================================#
        test_name   |ntup| tsamples |psamples|  p-value |Assessment
#=============================================================================#
   diehard_birthdays|   0|       100|     100|0.72231844|  PASSED  
      diehard_operm5|   0|   1000000|     100|0.00000147|   WEAK   
  diehard_rank_32x32|   0|     40000|     100|0.00000000|  FAILED  
    diehard_rank_6x8|   0|    100000|     100|0.00003364|   WEAK   
   diehard_bitstream|   0|   2097152|     100|0.88547776|  PASSED  
        diehard_opso|   0|   2097152|     100|0.00030434| 